In [1]:
import pandas as pd
import numpy as np
import os

df = pd.read_csv("processed/telco_eda_processed.csv")

print("Shape:", df.shape)
df.head()

Shape: (7043, 26)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,TenureGroup,MonthlyChargeGroup,NumberOfServices,HighValueCustomer,CustomerSegment
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,Yes,Electronic check,29.85,29.85,No,0-6 Months,Low,1,False,New Customer
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,Mailed check,56.95,1889.50,No,25-48 Months,Medium,3,False,Regular Customer
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,Yes,Mailed check,53.85,108.15,Yes,0-6 Months,Medium,3,False,New Customer
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,No,Bank transfer (automatic),42.30,1840.75,No,25-48 Months,Medium,3,False,Regular Customer
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,Yes,Electronic check,70.70,151.65,Yes,0-6 Months,High,1,False,New Customer


In [2]:
df["Churn"].value_counts()

Churn
No     5174
Yes    1869
Name: count, dtype: int64

In [3]:
df["Churn"] = df["Churn"].map({
    "No": 0,
    "Yes": 1
})

df["Churn"].value_counts()

Churn
0    5174
1    1869
Name: count, dtype: int64

In [4]:
df.isnull().sum().sort_values(ascending=False).head(10)

TotalCharges       11
customerID          0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
gender              0
dtype: int64

In [5]:
df["TotalCharges"] = df["TotalCharges"].fillna(
    df["TotalCharges"].median()
)

In [6]:
print("Total missing values:", df.isnull().sum().sum())

Total missing values: 0


In [7]:
customer_ids = df["customerID"].copy()

df = df.drop(columns=["customerID"])

In [8]:
df["NewCustomer"] = (df["tenure"] <= 6).astype(int)

df["LongTermCustomer"] = (df["tenure"] >= 48).astype(int)

In [9]:
df["ChargeToTenure"] = df["TotalCharges"] / (df["tenure"] + 1)

In [10]:
monthly_threshold = df["MonthlyCharges"].quantile(0.75)

df["HighMonthlyCharge"] = (
    df["MonthlyCharges"] >= monthly_threshold
).astype(int)

In [11]:
df["HighValueHighRisk"] = (
    (df["HighMonthlyCharge"] == 1) &
    (df["NewCustomer"] == 1)
).astype(int)

In [12]:
df[
    [
        "tenure",
        "MonthlyCharges",
        "TotalCharges",
        "NewCustomer",
        "LongTermCustomer",
        "ChargeToTenure",
        "HighMonthlyCharge",
        "HighValueHighRisk"
    ]
].head()

,tenure,MonthlyCharges,TotalCharges,NewCustomer,LongTermCustomer,ChargeToTenure,HighMonthlyCharge,HighValueHighRisk
0,1,29.85,29.85,1,0,14.925000,0,0
1,34,56.95,1889.50,0,0,53.985714,0,0
2,2,53.85,108.15,1,0,36.050000,0,0
3,45,42.30,1840.75,0,0,40.016304,0,0
4,2,70.70,151.65,1,0,50.550000,0,0


In [13]:
X = df.drop(columns=["Churn"])

y = df["Churn"]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (7043, 29)
y shape: (7043,)


In [14]:
#Identify categorical and numerical columns
categorical_columns = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_columns = X.select_dtypes(
    include=["int64", "float64", "int32", "float32"]
).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

print("\nNumerical columns:")
print(numerical_columns)

Categorical columns:
['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TenureGroup', 'MonthlyChargeGroup', 'CustomerSegment']

Numerical columns:
['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges', 'NumberOfServices', 'NewCustomer', 'LongTermCustomer', 'ChargeToTenure', 'HighMonthlyCharge', 'HighValueHighRisk']


/tmp/ipykernel_252762/2140889118.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_columns = X.select_dtypes(


In [15]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

Training samples: 5634
Testing samples: 1409


In [16]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

In [17]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

In [18]:
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]
)

In [19]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            numeric_transformer,
            numerical_columns
        ),
        (
            "cat",
            categorical_transformer,
            categorical_columns
        )
    ]
)

In [20]:
X_train_processed = preprocessor.fit_transform(X_train)

X_test_processed = preprocessor.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (5634, 65)
Processed testing shape: (1409, 65)


In [21]:
feature_names = preprocessor.get_feature_names_out()

print("Number of features:", len(feature_names))

print(feature_names[:20])

Number of features: 65
['num__SeniorCitizen' 'num__tenure' 'num__MonthlyCharges'
 'num__TotalCharges' 'num__NumberOfServices' 'num__NewCustomer'
 'num__LongTermCustomer' 'num__ChargeToTenure' 'num__HighMonthlyCharge'
 'num__HighValueHighRisk' 'cat__gender_Female' 'cat__gender_Male'
 'cat__Partner_No' 'cat__Partner_Yes' 'cat__Dependents_No'
 'cat__Dependents_Yes' 'cat__PhoneService_No' 'cat__PhoneService_Yes'
 'cat__MultipleLines_No' 'cat__MultipleLines_No phone service']


In [22]:
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_test_processed_df = pd.DataFrame(
    X_test_processed,
    columns=feature_names,
    index=X_test.index
)

X_train_processed_df.head()

,num__SeniorCitizen,num__tenure,num__MonthlyCharges,num__TotalCharges,num__NumberOfServices,num__NewCustomer,num__LongTermCustomer,num__ChargeToTenure,num__HighMonthlyCharge,num__HighValueHighRisk,...,cat__TenureGroup_7-12 Months,cat__MonthlyChargeGroup_High,cat__MonthlyChargeGroup_Low,cat__MonthlyChargeGroup_Medium,cat__MonthlyChargeGroup_Very High,cat__CustomerSegment_Long-Term Customer,cat__CustomerSegment_Loyal High-Value,cat__CustomerSegment_New Customer,cat__CustomerSegment_New High-Value,cat__CustomerSegment_Regular Customer
3738,-0.441773,0.102371,-0.521976,-0.263289,-0.184954,-0.511954,-0.696402,-0.236288,-0.583226,-0.132358,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
3151,-0.441773,-0.711743,0.337478,-0.504814,-0.667823,-0.511954,-0.696402,0.182960,-0.583226,-0.132358,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4860,-0.441773,-0.793155,-0.809013,-0.751213,-0.184954,-0.511954,-0.696402,-0.322843,-0.583226,-0.132358,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
3867,-0.441773,-0.263980,0.284384,-0.173699,0.780784,-0.511954,-0.696402,0.159364,-0.583226,-0.132358,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3810,-0.441773,-1.281624,-0.676279,-0.990851,-1.150692,1.953301,-0.696402,-0.660445,-0.583226,-0.132358,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


In [23]:
print("Training class distribution:")
print(y_train.value_counts())

print("\nTraining class percentage:")
print(
    y_train.value_counts(normalize=True).mul(100).round(2)
)

Training class distribution:
Churn
0    4139
1    1495
Name: count, dtype: int64

Training class percentage:
Churn
0    73.46
1    26.54
Name: proportion, dtype: float64


In [24]:
import joblib

os.makedirs("models", exist_ok=True)
os.makedirs("processed", exist_ok=True)

joblib.dump(
    preprocessor,
    "models/preprocessor.pkl"
)

joblib.dump(
    feature_names,
    "models/feature_names.pkl"
)

X_train_processed_df.to_csv(
    "processed/X_train_processed.csv",
    index=False
)

X_test_processed_df.to_csv(
    "processed/X_test_processed.csv",
    index=False
)

y_train.to_csv(
    "processed/y_train.csv",
    index=False
)

y_test.to_csv(
    "processed/y_test.csv",
    index=False
)

customer_ids.to_csv(
    "processed/customer_ids.csv",
    index=False
)

